# Economic Regime Change Detection via Boosting Models

## Objective
Extend the FOMC rate-decision forecasting framework to output **probability distributions** across three outcomes — `Lower`, `Same`, `Higher` — rather than a single hard prediction. This probabilistic framing enables:
- Identifying **economic regimes** (inflationary vs. contractionary) from the shape of the distribution
- Quantifying model **uncertainty** at each FOMC meeting
- Better calibration for downstream decision-making

**Features used** (all lagged to prevent leakage):
- WTI Crude Oil 30-day average (lags 1–4)
- Unemployment Rate — UNRATE (lags 1–4)
- CPI Year-over-Year % (lags 1–4)
- Previous Fed Funds Target Rate (lags 1–4)
- Derived trend / inertia features

## Test Methodology: Walk-Forward (Expanding Window) Validation
For each FOMC meeting at index $t$, starting from `INITIAL_TRAIN_SIZE = 40`:
1. Train on all meetings $[0, t)$
2. Predict **probability distribution** $[P(\text{Lower}), P(\text{Same}), P(\text{Higher})]$ for meeting $t$
3. Record distribution → advance $t$ by 1

## Models & Soft-Probability Method
| Model | Soft-probability method |
|---|---|
| **XGBoost** | `objective='multi:softprob'` → `predict_proba()` |
| **LightGBM** | `predict_proba()` |
| **CatBoost** | `predict_proba()` |
| **HistGradientBoosting** | `predict_proba()` |

Hyperparameters are **pre-tuned via Optuna** in `boosting_models.ipynb` and hardcoded here.

## Evaluation Metrics
| Metric | Purpose |
|---|---|
| Accuracy / F1 Macro | Hard-label quality (argmax of probs) |
| **Brier Score** | Probability calibration (lower = better) |
| **Log-Loss** | Entropy of predicted distributions (lower = better) |

> **Class imbalance note:** ~77 % of meetings are "Same", 15 % "Higher", 8 % "Lower".
> All models use balanced class weights. **F1 Macro** is the primary hard-label metric.


In [1]:
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, log_loss, brier_score_loss
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier

# ── Optional packages ─────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — skipping')

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM not installed — skipping')

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed — skipping')

print('All imports OK')
print(f'  XGBoost : {XGBOOST_AVAILABLE}')
print(f'  LightGBM: {LGBM_AVAILABLE}')
print(f'  CatBoost: {CATBOOST_AVAILABLE}')

All imports OK
  XGBoost : True
  LightGBM: True
  CatBoost: True


---
## 1. Data Loading & Feature Setup

In [2]:
_DATA_URL = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/df_model.csv'
df_model = pd.read_csv(_DATA_URL, parse_dates=['meeting_date'])

print(f'Loaded df_model: {df_model.shape[0]} rows × {df_model.shape[1]} columns')
print(f'Date range     : {df_model["meeting_date"].min().date()} → {df_model["meeting_date"].max().date()}')
print()
print('Decision distribution:')
print(df_model['decision'].value_counts())

Loaded df_model: 135 rows × 26 columns
Date range     : 2009-09-23 → 2026-01-28

Decision distribution:
decision
same      104
higher     20
lower      11
Name: count, dtype: int64


In [3]:
# ── Feature / Label Setup ─────────────────────────────────────────────────────
NON_FEATURE_COLS = ['meeting_date', 'decision', 'decision_num', 'prev_decision']
feature_cols = [c for c in df_model.columns if c not in NON_FEATURE_COLS]

X = df_model[feature_cols].values.astype(float)      # shape (N, ~22)
y_orig = df_model['decision_num'].values.astype(int)  # -1 = Lower, 0 = Same, 1 = Higher
meeting_dates = df_model['meeting_date'].values

# XGBoost / LightGBM / CatBoost require 0-indexed integer labels
LABEL_MAP     = {-1: 0,  0: 1,  1: 2}   # original → encoded
INV_LABEL_MAP = { 0: -1, 1: 0,  2: 1}   # encoded  → original
ENCODED_NAMES = { 0: 'Lower', 1: 'Same', 2: 'Higher'}
LABEL_VALS    = [-1, 0, 1]
LABEL_STRS    = ['Lower', 'Same', 'Higher']
# Encoded order: 0=Lower, 1=Same, 2=Higher

y_enc = np.array([LABEL_MAP[v] for v in y_orig])    # 0, 1, 2

print(f'Features: {len(feature_cols)}')
print(f'Samples : {len(X)}')
print(f'Label distribution (encoded): {dict(zip(*np.unique(y_enc, return_counts=True)))}')
print('  0=Lower, 1=Same, 2=Higher')

Features: 22
Samples : 135
Label distribution (encoded): {0: 11, 1: 104, 2: 20}
  0=Lower, 1=Same, 2=Higher


---
## 2. Best Hyperparameters (from Optuna in `boosting_models.ipynb`)

In [4]:
# ── Pre-tuned hyperparameters (Optuna, boosting_models.ipynb) ─────────────────
BEST_PARAMS = {
    'XGBoost': dict(
        n_estimators     = 471,
        max_depth        = 3,
        learning_rate    = 0.161042,
        subsample        = 0.712890,
        colsample_bytree = 0.847161,
        min_child_weight = 3,
        gamma            = 0.360274,
        reg_alpha        = 0.243105,
        reg_lambda       = 3.207902,
    ),
    'LightGBM': dict(
        n_estimators      = 500,
        num_leaves        = 42,
        learning_rate     = 0.056209,
        subsample         = 0.685067,
        colsample_bytree  = 0.849375,
        min_child_samples = 17,
        reg_alpha         = 0.658121,
        reg_lambda        = 3.342565,
    ),
    'CatBoost': dict(
        iterations          = 331,
        depth               = 2,
        learning_rate       = 0.018522,
        l2_leaf_reg         = 3.272855,
        bagging_temperature = 0.324958,
    ),
    'HistGradientBoosting': dict(
        max_iter          = 478,
        max_depth         = 4,
        learning_rate     = 0.036259,
        min_samples_leaf  = 3,
        l2_regularization = 2.391524,
        max_leaf_nodes    = 38,
    ),
}

print('Pre-tuned hyperparameters loaded:')
for name, params in BEST_PARAMS.items():
    print(f'  {name}: {params}')

Pre-tuned hyperparameters loaded:
  XGBoost: {'n_estimators': 471, 'max_depth': 3, 'learning_rate': 0.161042, 'subsample': 0.71289, 'colsample_bytree': 0.847161, 'min_child_weight': 3, 'gamma': 0.360274, 'reg_alpha': 0.243105, 'reg_lambda': 3.207902}
  LightGBM: {'n_estimators': 500, 'num_leaves': 42, 'learning_rate': 0.056209, 'subsample': 0.685067, 'colsample_bytree': 0.849375, 'min_child_samples': 17, 'reg_alpha': 0.658121, 'reg_lambda': 3.342565}
  CatBoost: {'iterations': 331, 'depth': 2, 'learning_rate': 0.018522, 'l2_leaf_reg': 3.272855, 'bagging_temperature': 0.324958}
  HistGradientBoosting: {'max_iter': 478, 'max_depth': 4, 'learning_rate': 0.036259, 'min_samples_leaf': 3, 'l2_regularization': 2.391524, 'max_leaf_nodes': 38}


---
## 3. Walk-Forward Validation Helpers (Probability Output)

In [5]:
INITIAL_TRAIN_SIZE = 40

def augment_missing_classes(X_train, y_train, all_classes=(0, 1, 2)):
    """
    Inject one synthetic sample per missing class (mean features, tiny weight).
    Ensures all 3 classes are always present — critical for multiclass softmax.
    Returns (X_aug, y_aug, sample_weights).
    """
    present = set(y_train)
    missing = set(all_classes) - present

    class_counts = np.bincount(y_train, minlength=3)
    n_total      = len(y_train)
    real_weights = np.array([n_total / (3 * max(class_counts[c], 1)) for c in y_train],
                             dtype=float)

    if not missing:
        return X_train, y_train, real_weights

    X_mean  = X_train.mean(axis=0, keepdims=True)
    X_synth = np.vstack([X_mean] * len(missing))
    y_synth = np.array(sorted(missing))
    w_synth = np.full(len(missing), 1e-9)

    return (np.vstack([X_train, X_synth]),
            np.concatenate([y_train, y_synth]),
            np.concatenate([real_weights, w_synth]))


def walk_forward_proba(
    fit_fn,
    proba_fn,
    X, y_enc,
    n_start=INITIAL_TRAIN_SIZE,
    verbose=True
):
    """
    Expanding-window walk-forward validation that collects probability distributions.

    Returns
    -------
    actuals_enc : np.array (N_test,)    — true encoded labels {0,1,2}
    probas      : np.array (N_test, 3)  — P(Lower), P(Same), P(Higher) per meeting
    """
    actuals, probas = [], []
    total = len(X) - n_start

    for t in range(n_start, len(X)):
        X_train, y_train = X[:t], y_enc[:t]
        X_test           = X[t:t+1]

        model  = fit_fn(X_train, y_train)
        p_dist = proba_fn(model, X_test)   # shape (1, 3)

        actuals.append(y_enc[t])
        probas.append(p_dist[0])           # shape (3,)

        done = t - n_start + 1
        if verbose and (done % 25 == 0 or done == total):
            p = p_dist[0]
            pred_enc = int(np.argmax(p))
            flag = '✓' if y_enc[t] == pred_enc else '✗'
            print(
                f'  [{done:3d}/{total}]  '
                f'{pd.Timestamp(meeting_dates[t]).date()}  '
                f'true={ENCODED_NAMES[y_enc[t]]:7s}  '
                f'pred={ENCODED_NAMES[pred_enc]:7s}  {flag}  '
                f'[L={p[0]:.2f} S={p[1]:.2f} H={p[2]:.2f}]'
            )

    return np.array(actuals), np.vstack(probas)   # (N,), (N, 3)


def report_prob_metrics(name, actuals_enc, probas):
    """Compute and print metrics for a probabilistic classifier. Returns a results dict."""
    preds_enc    = np.argmax(probas, axis=1)
    actuals_orig = np.array([INV_LABEL_MAP[a] for a in actuals_enc])
    preds_orig   = np.array([INV_LABEL_MAP[p] for p in preds_enc])

    acc    = accuracy_score(actuals_orig, preds_orig)
    f1_mac = f1_score(actuals_orig, preds_orig, labels=LABEL_VALS,
                      average='macro', zero_division=0)
    f1_wt  = f1_score(actuals_orig, preds_orig, labels=LABEL_VALS,
                      average='weighted', zero_division=0)

    y_onehot = label_binarize(actuals_enc, classes=[0, 1, 2])
    logloss  = log_loss(actuals_enc, probas, labels=[0, 1, 2])
    brier    = np.mean([
        brier_score_loss(y_onehot[:, k], probas[:, k])
        for k in range(3)
    ])

    n_test  = len(actuals_orig)
    n_right = int(acc * n_test)

    print('=' * 68)
    print(f'  {name}')
    print('=' * 68)
    print(f'  Accuracy     : {acc:.4f}  ({n_right}/{n_test} correct)')
    print(f'  F1 Macro     : {f1_mac:.4f}  (primary hard-label metric)')
    print(f'  F1 Weighted  : {f1_wt:.4f}')
    print(f'  Log-Loss     : {logloss:.4f}  (↓ better calibration)')
    print(f'  Brier Score  : {brier:.4f}   (↓ better calibration)')
    print()
    print(classification_report(actuals_orig, preds_orig, labels=LABEL_VALS,
                                  target_names=LABEL_STRS, zero_division=0))

    return dict(
        name=name,
        accuracy=acc, f1_macro=f1_mac, f1_weighted=f1_wt,
        log_loss=logloss, brier_score=brier,
        actuals_enc=actuals_enc, preds_enc=preds_enc,
        actuals_orig=actuals_orig, preds_orig=preds_orig,
        probas=probas,
    )


print('Helpers defined.')

Helpers defined.


---
## 4. Walk-Forward Probabilistic Evaluation

Each model returns `[P(Lower), P(Same), P(Higher)]` at every FOMC meeting using the pre-tuned hyperparameters.

In [6]:
final_results = {}

# ── XGBoost ───────────────────────────────────────────────────────────────────
if XGBOOST_AVAILABLE:
    t0 = time.time()
    bp = BEST_PARAMS['XGBoost']
    print('Running XGBoost walk-forward …')

    def xgb_fit(Xtr, ytr):
        Xa, ya, sw = augment_missing_classes(Xtr, ytr)
        m = XGBClassifier(
            **bp,
            objective='multi:softprob',
            num_class=3,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42, verbosity=0
        )
        m.fit(Xa, ya, sample_weight=sw)
        return m

    def xgb_proba(m, Xt):
        """Returns shape (1, 3): [P(Lower), P(Same), P(Higher)]"""
        return m.predict_proba(Xt)

    act_xgb, prob_xgb = walk_forward_proba(xgb_fit, xgb_proba, X, y_enc, verbose=False)
    r = report_prob_metrics('XGBoost', act_xgb, prob_xgb)
    final_results['XGBoost'] = r
    print(f'  ↳ done in {time.time()-t0:.1f}s\n')
else:
    print('XGBoost not available — skipping.')

Running XGBoost walk-forward …
  XGBoost
  Accuracy     : 0.6632  (62/95 correct)
  F1 Macro     : 0.6381  (primary hard-label metric)
  F1 Weighted  : 0.6738
  Log-Loss     : 1.0868  (↓ better calibration)
  Brier Score  : 0.1762   (↓ better calibration)

              precision    recall  f1-score   support

       Lower       0.58      0.64      0.61        11
        Same       0.85      0.61      0.71        64
      Higher       0.46      0.85      0.60        20

    accuracy                           0.66        95
   macro avg       0.63      0.70      0.64        95
weighted avg       0.74      0.66      0.67        95

  ↳ done in 67.6s



In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
if LGBM_AVAILABLE:
    t0 = time.time()
    bp = BEST_PARAMS['LightGBM']
    print('Running LightGBM walk-forward …')

    def lgbm_fit(Xtr, ytr):
        # augment_missing_classes ensures all 3 labels are present so
        # predict_proba always returns shape (1, 3) — not (1, 2) when
        # the rare 'Lower' class is absent from early training windows.
        Xa, ya, sw = augment_missing_classes(Xtr, ytr)
        m = LGBMClassifier(
            **bp,
            class_weight='balanced',
            random_state=42, verbose=-1, n_jobs=1
        )
        m.fit(Xa, ya, sample_weight=sw)
        return m

    def lgbm_proba(m, Xt):
        """Returns shape (1, 3): [P(Lower), P(Same), P(Higher)]"""
        return m.predict_proba(Xt)

    act_lgbm, prob_lgbm = walk_forward_proba(lgbm_fit, lgbm_proba, X, y_enc, verbose=False)
    r = report_prob_metrics('LightGBM', act_lgbm, prob_lgbm)
    final_results['LightGBM'] = r
    print(f'  ↳ done in {time.time()-t0:.1f}s\n')
else:
    print('LightGBM not available — skipping.')

In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────────
if CATBOOST_AVAILABLE:
    t0 = time.time()
    bp = BEST_PARAMS['CatBoost']
    print('Running CatBoost walk-forward …')

    def cat_fit(Xtr, ytr):
        Xa, ya, sw = augment_missing_classes(Xtr, ytr)
        m = CatBoostClassifier(**bp, random_seed=42, verbose=0, thread_count=1)
        m.fit(Xa, ya, sample_weight=sw)
        return m

    def cat_proba(m, Xt):
        """Returns shape (1, 3): [P(Lower), P(Same), P(Higher)]"""
        return m.predict_proba(Xt)

    act_cat, prob_cat = walk_forward_proba(cat_fit, cat_proba, X, y_enc, verbose=False)
    r = report_prob_metrics('CatBoost', act_cat, prob_cat)
    final_results['CatBoost'] = r
    print(f'  ↳ done in {time.time()-t0:.1f}s\n')
else:
    print('CatBoost not available — skipping.')

In [ ]:
# ── HistGradientBoosting ───────────────────────────────────────────────────────
t0 = time.time()
bp = BEST_PARAMS['HistGradientBoosting']
print('Running HistGradientBoosting walk-forward …')

def hgb_fit(Xtr, ytr):
    # augment_missing_classes ensures all 3 labels are present so
    # predict_proba always returns shape (1, 3).
    Xa, ya, sw = augment_missing_classes(Xtr, ytr)
    m = HistGradientBoostingClassifier(**bp, class_weight='balanced', random_state=42)
    m.fit(Xa, ya, sample_weight=sw)
    return m

def hgb_proba(m, Xt):
    """Returns shape (1, 3): [P(Lower), P(Same), P(Higher)]"""
    return m.predict_proba(Xt)

act_hgb, prob_hgb = walk_forward_proba(hgb_fit, hgb_proba, X, y_enc, verbose=False)
r = report_prob_metrics('HistGradientBoosting', act_hgb, prob_hgb)
final_results['HistGradientBoosting'] = r
print(f'  ↳ done in {time.time()-t0:.1f}s\n')

---
## 5. Model Comparison Summary

In [ ]:
rows = []
for r in final_results.values():
    rows.append({
        'Model'       : r['name'],
        'Accuracy'    : round(r['accuracy'],    4),
        'F1 Macro'    : round(r['f1_macro'],    4),
        'F1 Weighted' : round(r['f1_weighted'], 4),
        'Log-Loss'    : round(r['log_loss'],    4),
        'Brier Score' : round(r['brier_score'], 4),
    })

df_compare = pd.DataFrame(rows).sort_values('F1 Macro', ascending=False)
print(df_compare.to_string(index=False))

---
## 6. Probability Distribution Visualizations

In [ ]:
# ── Stacked bar: P(Lower) / P(Same) / P(Higher) over time ────────────────────
eval_dates = pd.to_datetime(meeting_dates[INITIAL_TRAIN_SIZE:])
COLORS = {'Lower': '#d62728', 'Same': '#aec7e8', 'Higher': '#2ca02c'}

fig, axes = plt.subplots(len(final_results), 1,
                          figsize=(16, 4 * len(final_results)),
                          sharex=True)
if len(final_results) == 1:
    axes = [axes]

for ax, (model_key, r) in zip(axes, final_results.items()):
    probs = r['probas']      # (N_test, 3): col 0=Lower, 1=Same, 2=Higher
    act   = r['actuals_enc']

    ax.bar(eval_dates, probs[:, 2], color=COLORS['Higher'], label='P(Higher)', width=15)
    ax.bar(eval_dates, probs[:, 1], bottom=probs[:, 2],
           color=COLORS['Same'], label='P(Same)', width=15)
    ax.bar(eval_dates, probs[:, 0], bottom=probs[:, 2] + probs[:, 1],
           color=COLORS['Lower'], label='P(Lower)', width=15)

    # True decision markers above the bar
    marker_map = {0: ('v', 'red'), 1: ('o', 'grey'), 2: ('^', 'green')}
    for enc, (mk, mc) in marker_map.items():
        idx = act == enc
        ax.scatter(eval_dates[idx], np.ones(idx.sum()) * 1.02,
                   marker=mk, color=mc, s=40, zorder=5, clip_on=False)

    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Probability')
    ax.set_title(r['name'])
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

axes[-1].set_xlabel('FOMC Meeting Date')
fig.suptitle('Walk-Forward Probability Distributions — All Models\n'
             'Markers: ▽=Lower  ○=Same  △=Higher (actual outcome)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predictive Entropy over time ──────────────────────────────────────────────
# High entropy ≈ uncertain regime transition; Low entropy ≈ confident call

fig, axes = plt.subplots(len(final_results), 1,
                          figsize=(16, 3 * len(final_results)),
                          sharex=True)
if len(final_results) == 1:
    axes = [axes]

for ax, (model_key, r) in zip(axes, final_results.items()):
    probs   = r['probas'].clip(1e-9, 1)
    entropy = -np.sum(probs * np.log(probs), axis=1)

    ax.fill_between(eval_dates, entropy, alpha=0.4, color='steelblue')
    ax.plot(eval_dates, entropy, color='steelblue', linewidth=0.8)
    ax.axhline(np.log(3), color='red', linestyle='--', linewidth=0.8,
               label=f'Max entropy = {np.log(3):.2f} (uniform)')
    ax.set_ylabel('Entropy (nats)')
    ax.set_title(f'{r["name"]} — Predictive Uncertainty')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('FOMC Meeting Date')
fig.suptitle('Predictive Entropy: High = Uncertain Regime, Low = Confident',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Human-readable probability table: last 15 meetings ───────────────────────
N_SHOW = 15

for model_key, r in final_results.items():
    probs = r['probas']
    acts  = r['actuals_enc']
    preds = r['preds_enc']

    print(f'\n{r["name"]} — Last {N_SHOW} meetings')
    print(f'{"Date":<14} {"Actual":<8} {"Pred":<8}'
          f' {"P(Lower)":>9} {"P(Same)":>9} {"P(Higher)":>10}  OK?')
    print('-' * 72)

    for i in range(max(0, len(probs) - N_SHOW), len(probs)):
        flag = '✓' if acts[i] == preds[i] else '✗'
        print(
            f'{str(eval_dates[i].date()):<14}'
            f' {ENCODED_NAMES[acts[i]]:<8}'
            f' {ENCODED_NAMES[preds[i]]:<8}'
            f' {probs[i,0]:>8.1%}'
            f' {probs[i,1]:>9.1%}'
            f' {probs[i,2]:>9.1%}'
            f'   {flag}'
        )

---
## 7. Calibration Check (Reliability Diagram)

In [ ]:
fig, axes = plt.subplots(len(final_results), 3,
                          figsize=(14, 4 * len(final_results)),
                          sharey=True)
if len(final_results) == 1:
    axes = [axes]

class_colors = ['#d62728', '#aec7e8', '#2ca02c']

for row_axes, (model_key, r) in zip(axes, final_results.items()):
    probs    = r['probas']
    act_enc  = r['actuals_enc']
    y_onehot = label_binarize(act_enc, classes=[0, 1, 2])

    for k, (ax, cls_name, color) in enumerate(zip(row_axes, LABEL_STRS, class_colors)):
        try:
            frac_pos, mean_pred = calibration_curve(
                y_onehot[:, k], probs[:, k], n_bins=7, strategy='quantile'
            )
            ax.plot(mean_pred, frac_pos, 's-', color=color, label='Model', linewidth=1.5)
        except ValueError:
            pass

        ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Perfect')
        ax.set_title(f'{r["name"]}\n{cls_name}')
        ax.set_xlabel('Mean Predicted Prob')
        ax.set_ylabel('Fraction Positive')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

fig.suptitle('Reliability Diagrams (Calibration)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()